# LLM summary
This notebook showcases a simple engineered prompt used to summarize a long piece of text given in a pdf format.

In [9]:
from langchain_community.chat_models import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

import pytesseract
from pdf2image import pdfinfo_from_path, convert_from_path
from tqdm.notebook import tqdm

Make sure to run an ollama server locally somewhere prior to this step via `ollama serve`

In [10]:
local_llm = "llama3.2"
llm = ChatOllama(model=local_llm, temperature=0)

We then load in the entire PDF, which is likely to be well over the LLM's context limit. The code below assumes each pdf's page to be static images, using OCR to extract textual information.

In [11]:
pdf_file = 'test-documents/the-vampyre.pdf'
info = pdfinfo_from_path(pdf_file, userpw=None, poppler_path=None)

maxPages = info["Pages"]
page_data = []
for page in tqdm(range(1, maxPages+1)):
    page_img = convert_from_path(pdf_file, dpi=200, first_page=page, last_page = page)[0]
    page_text = pytesseract.image_to_string(page_img)
    page_data.append(page_text)

  0%|          | 0/25 [00:00<?, ?it/s]

100%|██████████| 25/25 [04:47<00:00, 11.51s/it]


Option 1: summarize each page (or group of pages not exceeding the token limit), then concatenate any future story to the summary

In [37]:
summary = '''
Characters:
    - (name) (description) (personality)
    (only list main characters)
Plot Overview:
    Summarize, in short, the story up to the current point in the story so that subsequent reader may pick up from here.
    Must be in one concise paragraph, not bullet points. Max 5 sentences.
Key events:
    - list key events up to this point as bullet points (i.e. A meeting B, C dying)
    make sure to mention the person (who), action (what), and place (where) if said information can be found
    (at most 20 events, concatenate existing ones once more story is added)
Observations:
    - list interesting traits of each characters, as well as events that leads to such an observation. 
    (at most 20 events, concatenate existing ones once more story is added)
Conclusion:
    - if the story were to end here, what would you describe of the ending?
'''
summary_format = summary
with open('prompts/story_summary.txt', 'r') as file:
    prompt = file.read()

In [38]:
### Generate

# Prompt
prompt = PromptTemplate(
    template=prompt,
    input_variables=["summary", "summary_format", "story"],
)

# Chain
rag_chain = prompt | llm | StrOutputParser()

# Run
for page in tqdm(page_data):
    summary = rag_chain.invoke({"summary": summary, "summary_format": summary_format, "story":page})

 12%|█▏        | 3/25 [00:24<02:58,  8.11s/it]


KeyboardInterrupt: 

In [39]:
print(summary)

Characters:
    - Lord Ruthven (nobleman with singularities, dead grey eye, beautiful but pale face) (mysterious and intimidating)
    - Aubrey (young gentleman with high romantic feeling of honour and candour)

Plot Overview:
Lord Ruthven appears at various parties in London, causing a sensation due to his unusual behavior and striking appearance. He seems out of place among the lively atmosphere, and those who meet him feel a sense of awe or fear. Despite this, he is invited to every house, and women try to win his attention.

Key events:
    - Lord Ruthven appears at various parties in London (where), causing a sensation due to his unusual behavior and striking appearance
    - He gazes upon the mirth around him with a dead grey eye that seems to pierce through to the heart of those he looks at (who)
    - The female guests try to win his attention, but he remains aloof (what)
    - A young gentleman named Aubrey arrives in London (where), an orphan left with only a sister and great

In [ ]:
print(summary)